# Agentic AI, Day 4 Lab: Memory and Retrieval (RAG)

Your agent can reason and act, but it has two gaps: it forgets everything between calls, and it only knows what was in its training data. Today you close both.

1. **Memory:** keep a conversation in context, and keep it small as it grows.
2. **Retrieval (RAG):** look up real knowledge from your own documents and answer from it.

Everything runs locally on Ollama. For retrieval we use Ollama's own embeddings plus a little numpy, so there is **no vector database to install**.

Run the cells in order, top to bottom.

## Setup

You need a chat model (`llama3.1:8b` or `qwen2.5:7b`) and an embedding model. Pull the embedding model once, in a terminal:

```
ollama pull nomic-embed-text
```

And install numpy in the notebook if you do not have it:

In [ ]:
%pip install -q numpy

Now import everything and confirm both models are ready. **If your chat model is not `llama3.1`, change `MODEL` below.**

In [ ]:
import ollama
import numpy as np

MODEL = "llama3.1"                 # a chat model you have pulled
EMBED_MODEL = "nomic-embed-text"   # run: ollama pull nomic-embed-text

def ask(prompt: str) -> str:
    """Send one prompt to the chat model and return the reply text."""
    r = ollama.chat(model=MODEL, messages=[{"role": "user", "content": prompt}])
    return r.message.content

# confirm the embedding model is available
try:
    _ = ollama.embed(model=EMBED_MODEL, input="hello")
    print("Ready. Chat model:", MODEL, "  Embedding model:", EMBED_MODEL)
except Exception as e:
    print("Embedding model not ready. In a terminal run: ollama pull", EMBED_MODEL)
    print("Details:", e)

## Milestone 1: Conversation memory

The model is stateless: on its own it remembers nothing. Memory is simply the list of messages you keep and resend. To give the agent a memory of the conversation, you append each turn to a list and send the whole list every call.

We store both the user turn and the assistant reply as plain dictionaries, so the history is easy to read and reuse later.

In [ ]:
def chat(history, user_msg):
    """Append the turn, send the full history, store the reply."""
    history.append({"role": "user", "content": user_msg})
    reply = ollama.chat(model=MODEL, messages=history).message.content
    history.append({"role": "assistant", "content": reply})
    return reply

convo = []
print(chat(convo, "Hi, my name is Priya and I love hiking."))
print()
print(chat(convo, "What is my name, and what do I love?"))
print("\n(messages stored so far:", len(convo), ")")

**What you should see:** the second answer correctly says Priya and hiking. It can only do that because the first turn is still in the list we sent. That is memory: nothing more than keeping the history.

**Your turn:** add a third turn that refers back to something from the first, and confirm the model still has it.

## Milestone 2: Sliding window

Appending forever eventually overflows the context window. The simplest fix is a sliding window: keep only the last few turns and let older ones fall off. The cost is that the agent forgets anything older than the window.

We will set a small window on purpose, so you can watch it forget.

In [ ]:
def chat_window(history, user_msg, keep_turns=2):
    """Like chat, but only the last keep_turns turns are sent to the model."""
    history.append({"role": "user", "content": user_msg})
    recent = history[-(2 * keep_turns):]      # each turn is a user + assistant pair
    reply = ollama.chat(model=MODEL, messages=recent).message.content
    history.append({"role": "assistant", "content": reply})
    return reply

convo = []
chat_window(convo, "My name is Priya.")
chat_window(convo, "I have a dog named Bruno.")
chat_window(convo, "My favourite colour is teal.")
print(chat_window(convo, "What is my name?"))   # the name turn is now outside the window

**What you should see:** the agent probably cannot answer, because with `keep_turns=2` only the two most recent turns are sent, and the turn where you gave your name has slid out of the window.

**Your turn:** raise `keep_turns` to 4 and run it again. Now the name is back inside the window and the agent remembers. This is the whole trade-off of a sliding window: cheap and simple, but it forgets.

## Milestone 3: Summary memory

A sliding window forgets old facts. A better approach keeps the recent turns verbatim and folds the older ones into a short running summary. The agent then sees a small prompt that still carries the gist of everything that came before.

We keep a little state: the recent turns, plus a summary string that we update as old turns scroll off.

In [ ]:
def chat_smart(state, user_msg, keep_turns=2):
    """Send a running summary plus the last keep_turns turns."""
    state["recent"].append({"role": "user", "content": user_msg})

    msgs = [{"role": "system", "content": "Summary of earlier conversation: " + state["summary"]}]
    msgs += state["recent"][-(2 * keep_turns):]
    reply = ollama.chat(model=MODEL, messages=msgs).message.content
    state["recent"].append({"role": "assistant", "content": reply})

    # if we now have more than the window, fold the oldest turns into the summary
    if len(state["recent"]) > 2 * keep_turns:
        old = state["recent"][:-(2 * keep_turns)]
        state["summary"] = ask(
            "Update this summary with the key facts from the new turns. Keep it to 2 lines.\n"
            f"Old summary: {state['summary']}\nNew turns: {old}"
        )
        state["recent"] = state["recent"][-(2 * keep_turns):]
    return reply

state = {"recent": [], "summary": "none yet"}
chat_smart(state, "My name is Priya and I love hiking.")
chat_smart(state, "I have a dog named Bruno.")
chat_smart(state, "My favourite colour is teal.")
chat_smart(state, "I work as a teacher.")
print(chat_smart(state, "What is my name and what do I love?"))
print("\nRunning summary now:\n", state["summary"])

**What you should see:** even though the name turn scrolled out of the recent window, the agent still answers correctly, because that fact was folded into the running summary. Print the summary to see what the agent is carrying forward.

This is sliding window and summarization working together, which is what most real conversational agents use.

## Milestone 4: Embeddings

Now we switch to retrieval. The new idea is the **embedding**: a function that turns a piece of text into a vector of numbers, where texts with similar meaning get vectors that sit close together.

We measure closeness with **cosine similarity**: a number from about -1 to 1, where higher means more related. Let us prove that related words score higher than unrelated ones.

In [ ]:
def embed(text: str) -> np.ndarray:
    """Turn text into a vector using the local embedding model."""
    r = ollama.embed(model=EMBED_MODEL, input=text)
    return np.array(r.embeddings[0])

def cosine(a: np.ndarray, b: np.ndarray) -> float:
    """How close two vectors point. Higher means more similar in meaning."""
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

cat = embed("cat")
kitten = embed("kitten")
car = embed("automobile")

print("cat vs kitten    :", round(cosine(cat, kitten), 3))
print("cat vs automobile:", round(cosine(cat, car), 3))
print("vector length    :", len(cat), "numbers")

**What you should see:** cat and kitten score noticeably higher than cat and automobile. The model has placed similar meanings near each other, and the vector is a few hundred numbers long. This is the whole basis of semantic search: to find text related to a question, we embed both and compare.

**Your turn:** try your own pairs, for example "Paris" vs "France" and "Paris" vs "banana".

## Milestone 5: A mini RAG

Now build retrieval over your own documents. The recipe: chunk the documents, embed each chunk once, then for a question embed it, find the closest chunks, and answer using only those.

Here our documents are a few short facts about a fictional office. In a real system these would be chunks of your manuals, notes, or wiki.

In [ ]:
docs = [
    "The Trinity office is open from 9am to 6pm on weekdays.",
    "Employees get 24 days of paid leave per year.",
    "The cafeteria serves lunch between 12pm and 2pm.",
    "Remote work is allowed up to three days per week.",
    "The IT helpdesk can be reached at extension 4500.",
]

vecs = [embed(d) for d in docs]      # embed each chunk once (the index)

def search(query: str, k: int = 2):
    """Return the k chunks most similar to the query."""
    q = embed(query)
    sims = [cosine(q, v) for v in vecs]
    order = sorted(range(len(docs)), key=lambda i: sims[i], reverse=True)
    return [docs[i] for i in order[:k]]

def rag_answer(question: str, k: int = 2):
    """Retrieve relevant chunks, then answer using only them."""
    context = "\n".join(search(question, k))
    prompt = (
        "Answer using only the context below. If the answer is not in the "
        "context, say you do not know.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )
    return ask(prompt), context

answer, used = rag_answer("How many days of leave do I get?")
print("retrieved context:\n", used)
print("\nanswer:", answer)

**What you should see:** the retrieved context is the leave-related fact (and one other), and the answer says 24 days, grounded in that text.

**Your turn:** ask about lunch hours or remote work and watch the retrieved chunk change to match. Then ask something the docs do not cover, like "what is the wifi password", and see the agent say it does not know, instead of inventing one.

## Milestone 6: Grounded or not

The point of RAG in one comparison: ask the same question with and without retrieval, and see the difference.

In [ ]:
question = "How many vacation days do employees get, and what are the office hours?"

print("WITHOUT retrieval (the model guesses):")
print(ask(question))

print("\nWITH retrieval (grounded in the docs):")
answer, _ = rag_answer(question, k=3)
print(answer)

**What you should see:** without retrieval the model either refuses or makes up plausible numbers, because it has never seen this office. With retrieval it gives the real answer, 24 days and 9am to 6pm, drawn from the documents.

That gap, guessing versus grounded, is why RAG is the most common way real products put an LLM on top of private data.

## You did it

Today you gave the agent memory and a library:

- **Conversation memory:** keep and resend the history.
- **Sliding window:** cap to recent turns, accept that it forgets.
- **Summary memory:** fold old turns into a running summary to stay small.
- **Embeddings:** turn meaning into vectors and measure closeness.
- **Mini RAG:** chunk, embed, retrieve the closest chunks, answer from them.
- And you saw **grounded beat guessing** on a question about private data.

A key realisation: long-term memory and RAG are the same trick. Searching your past conversations for relevant pieces is just retrieval applied to history instead of documents.

### Optional challenges

1. **Show the source:** have `rag_answer` also print which chunk each fact came from.
2. **Chunk size:** split a longer paragraph into two-sentence chunks and see how retrieval changes.
3. **Top-k:** compare answers with k of 1, 2, and 4. More context is not always better.
4. **Memory plus RAG:** combine `chat_smart` with `rag_answer` so the agent both remembers the conversation and looks facts up.

### What comes next

Tomorrow, Day 5, is the finale: **multi-agent systems** where several agents work together, **guardrails** to keep them safe, and a **capstone** that ties the whole week together.